# Grounding DINO → SAM2 su video termico ferroviario

Pilot zero-shot senza fine-tuning. Il prompt testuale `person` produce box con Grounding DINO; le box selezionate inizializzano il tracking multi-oggetto di SAM 2.1.

In [ ]:
!nvidia-smi
%cd /content
!test -d sam2 || git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!pip install -q -e .
!pip install -q -U "transformers>=4.49,<5" accelerate safetensors
!mkdir -p checkpoints
!test -f checkpoints/sam2.1_hiera_small.pt || wget -q --show-progress -O checkpoints/sam2.1_hiera_small.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt

Caricare `san_donato_pilot_85_105.mp4`, clip locale da 20 secondi.

In [ ]:
%cd /content
from google.colab import files
uploaded = files.upload()

import cv2, os, shutil
from PIL import Image

video_path = "/content/san_donato_pilot_85_105.mp4"
frames_dir = "/content/san_donato_frames"
if os.path.exists(frames_dir): shutil.rmtree(frames_dir)
os.makedirs(frames_dir)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = 0
while True:
    ok, frame = cap.read()
    if not ok: break
    cv2.imwrite(f"{frames_dir}/{frame_count:05d}.jpg", frame)
    frame_count += 1
cap.release()
first_frame = Image.open(f"{frames_dir}/00000.jpg").convert("RGB")
print("Frame:", frame_count, "FPS:", fps)

In [ ]:
%cd /content/sam2
import torch
from sam2.build_sam import build_sam2_video_predictor
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

predictor = build_sam2_video_predictor(
    "configs/sam2.1/sam2.1_hiera_s.yaml",
    "/content/sam2/checkpoints/sam2.1_hiera_small.pt",
    device="cuda",
)
inference_state = predictor.init_state(video_path=frames_dir, offload_video_to_cpu=True)

gdino_model_id = "IDEA-Research/grounding-dino-tiny"
gdino_processor = AutoProcessor.from_pretrained(gdino_model_id)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(gdino_model_id).to("cuda").eval()
print("Modelli caricati")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

gdino_inputs = gdino_processor(images=first_frame, text=[["person"]], return_tensors="pt").to("cuda")
with torch.no_grad():
    gdino_outputs = gdino_model(**gdino_inputs)

result = gdino_processor.post_process_grounded_object_detection(
    gdino_outputs,
    gdino_inputs.input_ids,
    threshold=0.20,
    text_threshold=0.15,
    target_sizes=[(first_frame.height, first_frame.width)],
)[0]

detected_boxes = result["boxes"].cpu().numpy()
detected_scores = result["scores"].cpu().numpy()
keep = detected_scores >= 0.30
gdino_boxes_for_sam = detected_boxes[keep]
gdino_scores_for_sam = detected_scores[keep]
print("Rilevamenti totali:", len(detected_boxes), "selezionati:", len(gdino_boxes_for_sam))

In [ ]:
first_frame_array = np.array(first_frame)
plt.figure(figsize=(10, 8)); plt.imshow(first_frame_array)
axis = plt.gca()
for index, (box, score) in enumerate(zip(gdino_boxes_for_sam, gdino_scores_for_sam), start=1):
    x1, y1, x2, y2 = box
    axis.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, edgecolor="lime", facecolor="none", linewidth=3))
    axis.text(x1, max(12, y1-5), f"person {index}: {score:.2f}", color="lime", backgroundcolor="black")
plt.title("Grounding DINO — prompt person"); plt.axis("off")
plt.savefig("/content/gdino_person_detections.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
predictor.reset_state(inference_state)
with torch.inference_mode():
    for object_id, box in enumerate(gdino_boxes_for_sam, start=1):
        predictor.add_new_points_or_box(
            inference_state=inference_state,
            frame_idx=0,
            obj_id=object_id,
            box=box.astype(np.float32),
        )
print("Oggetti inizializzati:", len(gdino_boxes_for_sam))

In [ ]:
from tqdm.auto import tqdm
grounded_video_segments = {}
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    for frame_idx, object_ids, mask_logits in tqdm(
        predictor.propagate_in_video(inference_state), total=frame_count
    ):
        grounded_video_segments[frame_idx] = {
            int(object_id): (mask_logits[index] > 0).cpu().numpy().squeeze()
            for index, object_id in enumerate(object_ids)
        }
print("Frame elaborati:", len(grounded_video_segments))

In [ ]:
colors = {1: [1, 0, 0, 0.55], 2: [0, 1, 0, 0.55]}
check_frames = [0, 100, 200, 300, 400, 499]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for axis, frame_idx in zip(axes.ravel(), check_frames):
    frame = np.array(Image.open(f"{frames_dir}/{frame_idx:05d}.jpg").convert("RGB"))
    axis.imshow(frame)
    for object_id, mask in grounded_video_segments[frame_idx].items():
        overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
        overlay[mask] = colors[object_id]
        axis.imshow(overlay)
    axis.set_title(f"Frame {frame_idx} — {frame_idx/fps:.1f} s"); axis.axis("off")
plt.tight_layout()
plt.savefig("/content/grounded_sam2_six_frames.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
import json, zipfile, pandas as pd
start_frame_originale = 2125
object_ids = sorted(grounded_video_segments[0])
track_rows, detection_rows = [], []
mask_stack = np.zeros((frame_count, len(object_ids), first_frame.height, first_frame.width), dtype=np.uint8)

for object_id, box, score in zip(object_ids, gdino_boxes_for_sam, gdino_scores_for_sam):
    detection_rows.append({"clip_frame": 0, "source_frame": start_frame_originale, "object_id": object_id, "text_prompt": "person", "score": float(score), "xtl": float(box[0]), "ytl": float(box[1]), "xbr": float(box[2]), "ybr": float(box[3])})

for clip_frame in range(frame_count):
    for pos, object_id in enumerate(object_ids):
        mask = grounded_video_segments[clip_frame][object_id].astype(np.uint8)
        mask_stack[clip_frame, pos] = mask
        y, x = np.where(mask > 0)
        if len(x):
            track_rows.append({"frame": start_frame_originale+clip_frame, "clip_frame": clip_frame, "object_id": object_id, "label": "anomalia", "detected_class": "person", "score": float(gdino_scores_for_sam[pos]), "xtl": int(x.min()), "ytl": int(y.min()), "xbr": int(x.max())+1, "ybr": int(y.max())+1, "prompt": "person", "method": "grounding-dino-tiny+sam2.1-hiera-small", "mask_area_pixels": int(mask.sum()), "score_note": "score detector al frame 0"})

pd.DataFrame(detection_rows).to_csv("/content/gdino_frame0_detections.csv", index=False)
pd.DataFrame(track_rows).to_csv("/content/grounded_sam2_tracks.csv", index=False)
np.savez_compressed("/content/grounded_sam2_masks.npz", masks=mask_stack, object_ids=np.array(object_ids), source_frames=np.arange(start_frame_originale, start_frame_originale+frame_count))
metadata = {"experiment_id": "grounded_sam2_pilot_02", "source_video": "test_video_san_donato.mp4", "clip_seconds": [85,105], "text_prompt": "person", "grounding_dino_model": "IDEA-Research/grounding-dino-tiny", "detection_threshold": 0.20, "text_threshold": 0.15, "selection_threshold": 0.30, "sam2_checkpoint": "sam2.1_hiera_small.pt", "fine_tuning": False}
with open("/content/metadata.json", "w") as f: json.dump(metadata, f, indent=2)

mp4v_path = "/content/grounded_sam2_overlay_mp4v.mp4"
writer = cv2.VideoWriter(mp4v_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (first_frame.width, first_frame.height))
colors_bgr = {1: np.array([0,0,255], dtype=np.float32), 2: np.array([0,255,0], dtype=np.float32)}
for frame_idx in range(frame_count):
    frame = cv2.imread(f"{frames_dir}/{frame_idx:05d}.jpg")
    for object_id, mask in grounded_video_segments[frame_idx].items():
        mask = mask.astype(bool); color = colors_bgr[object_id]
        frame[mask] = (0.45*frame[mask].astype(np.float32) + 0.55*color).astype(np.uint8)
    writer.write(frame)
writer.release()
!ffmpeg -y -loglevel error -i /content/grounded_sam2_overlay_mp4v.mp4 -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p -movflags +faststart /content/grounded_sam2_overlay_h264.mp4

bundle = "/content/grounded_sam2_pilot_02_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path, name in [("/content/grounded_sam2_overlay_h264.mp4","grounded_sam2_overlay.mp4"), ("/content/gdino_frame0_detections.csv","gdino_frame0_detections.csv"), ("/content/grounded_sam2_tracks.csv","grounded_sam2_tracks.csv"), ("/content/grounded_sam2_masks.npz","grounded_sam2_masks.npz"), ("/content/metadata.json","metadata.json"), ("/content/gdino_person_detections.png","gdino_person_detections.png"), ("/content/grounded_sam2_six_frames.png","grounded_sam2_six_frames.png")]:
        archive.write(path, name)
print("Pacchetto:", bundle)
files.download(bundle)